# Commodity Open 2026: explore the pack

This notebook opens the Commodity Open data pack and shows how the point-in-time client decides what a model may see at a cutoff.

You need the pack (a directory with `MANIFEST.json`) and the `swing-dataset` package installed (`pip install -r requirements.txt` in the kit). Point `SWING_PACK_ROOT` at the pack directory, or place the pack at `../pack` relative to this notebook.

The pack is read only. Nothing here writes to it.

In [1]:
import json
import os
from datetime import UTC, datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd

from swing_dataset.pack.pit import PointInTime

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

PACK_ROOT = Path(os.environ.get("SWING_PACK_ROOT", "../pack")).resolve()
IST = ZoneInfo("Asia/Kolkata")
assert (PACK_ROOT / "MANIFEST.json").exists(), "set SWING_PACK_ROOT to the directory that holds MANIFEST.json"

manifest = json.loads((PACK_ROOT / "MANIFEST.json").read_text())
print("pack version:", manifest["version"])
print("built:", manifest["built_utc"])
print("lanes:", {lane: status["status"] for lane, status in manifest["lanes"].items()})
print("files:", len(manifest["files"]))

pack version: v2.1
built: 2026-10-23T22:00:45.911941+00:00
lanes: {'calendar': 'present', 'labels': 'present', 'macro': 'present', 'mcx': 'present', 'pack': 'present', 'swings': 'present'}
files: 32


## Every file and its clock

`MANIFEST.json` lists each parquet file with its sha256, row count and clock quality. The three clock classes:

| class | meaning | column the point-in-time client compares with the cutoff |
|---|---|---|
| `exact` | the fetch time was recorded at capture, so availability is known to the second | the exact clock the file's sidecar names: `fetched_utc` in the quote and news files, `captured_utc` in `calendar/release_vintages` |
| `published` | the source states a release instant; availability is at or after it | the published clock the sidecar names: `available_utc`, and `release_utc` in `calendar/release_schedule` |
| `unvintaged` | no credible availability time | excluded unless you opt in |

A file can be `mixed`, in which case every row carries its own `clock_quality`. The table below prints the clock columns each file's sidecar names.

In [2]:
files = pd.DataFrame(
    [
        {
            "path": f["path"],
            "rows": f["rows"],
            "mb": round(f["bytes"] / 1e6, 2),
            "clock": f["clock_quality"]["file"],
            "exact_clock": f["clock_quality"]["clocks"].get("exact"),
            "published_clock": f["clock_quality"]["clocks"].get("published"),
            "unvintaged_rows": (f["clock_quality"].get("rows") or {}).get("unvintaged", 0),
        }
        for f in manifest["files"]
    ]
)
files

,path,rows,mb,clock,exact_clock,published_clock,unvintaged_rows
0,calendar/release_schedule.parquet,3632,0.10,mixed,NaN,release_utc,554
1,calendar/release_surprises.parquet,1879,0.06,published,NaN,available_utc,0
2,calendar/release_vintages.parquet,100121,0.87,mixed,captured_utc,available_utc,1893
3,labels/chain_daily.parquet,205275,6.31,published,NaN,available_utc,0
4,labels/designations.parquet,17724,0.24,published,NaN,available_utc,0
5,labels/return_targets.parquet,17551,0.49,published,NaN,available_utc,0
6,labels/swing_labels.parquet,52653,0.54,published,NaN,available_utc,0
7,macro/alfred_vintages.parquet,122782,0.94,mixed,NaN,available_utc,39
8,macro/cot.parquet,3508,1.21,published,NaN,available_utc,0
9,macro/cot_legacy.parquet,3508,0.85,published,NaN,available_utc,0


## One cutoff

The task cutoff is 08:30 Asia/Kolkata on every MCX trading day. `PointInTime(root).available(asof)` returns one lazy frame per file; collecting it yields only rows whose clock is strictly before `asof`.

Take the cutoff of Friday 4 September 2026. In UTC that is 03:00.

In [3]:
pit = PointInTime(PACK_ROOT)
cutoff = datetime(2026, 9, 4, 8, 30, tzinfo=IST)
print("cutoff:", cutoff.isoformat(), "=", cutoff.astimezone(UTC).isoformat())
frames = pit.available(cutoff)

daily = frames["mcx/mcx_daily"].collect(columns=["session_date", "asset", "contract_id", "settlement", "volume_lots", "available_utc"])
daily = daily[daily["asset"].notna()]
print("mcx_daily rows for the four assets:", len(daily))
print("latest session visible:", daily["session_date"].max(), "(its close instant:", daily["available_utc"].max().isoformat(), ")")
daily[daily["session_date"] == daily["session_date"].max()].sort_values(["asset", "volume_lots"], ascending=[True, False]).groupby("asset").head(2)

cutoff: 2026-09-04T08:30:00+05:30 = 2026-09-04T03:00:00+00:00
mcx_daily rows for the four assets: 31254
latest session visible: 2026-09-03 (its close instant: 2026-09-03T18:00:00+00:00 )


,session_date,asset,contract_id,settlement,volume_lots,available_utc
153803,2026-09-03,copper,mcx:COPPER30SEP26FUT,1382.3,7478,2026-09-03 18:00:00+00:00
153804,2026-09-03,copper,mcx:COPPER30OCT26FUT,1389.9,734,2026-09-03 18:00:00+00:00
153818,2026-09-03,crude,mcx:CRUDEOIL21SEP26FUT,8643.0,62831,2026-09-03 18:00:00+00:00
153819,2026-09-03,crude,mcx:CRUDEOIL19OCT26FUT,8355.0,4701,2026-09-03 18:00:00+00:00
153834,2026-09-03,gold,mcx:GOLD05OCT26FUT,155775.0,10529,2026-09-03 18:00:00+00:00
153835,2026-09-03,gold,mcx:GOLD04DEC26FUT,157475.0,1438,2026-09-03 18:00:00+00:00
153901,2026-09-03,silver,mcx:SILVER04DEC26FUT,242349.0,10708,2026-09-03 18:00:00+00:00
153902,2026-09-03,silver,mcx:SILVER05MAR27FUT,248720.0,344,2026-09-03 18:00:00+00:00


The designation for this cutoff is itself a pack row (`labels/designations`), available at the close of the previous session. It names the contract a forecast is about, the anchor price `p_ref`, and the volatility scale `sigma_c`.

In [4]:
designations = frames["labels/designations"].collect(
    columns=["asset", "cutoff_utc", "horizon", "eligible", "contract_id", "p_ref", "sigma_c", "roll_k", "roll_contract_id", "available_utc"]
)
at_cutoff = designations[designations["cutoff_utc"] == pd.Timestamp(cutoff)]
at_cutoff.sort_values(["asset", "horizon"])

,asset,cutoff_utc,horizon,eligible,contract_id,p_ref,sigma_c,roll_k,roll_contract_id,available_utc
10100,copper,2026-09-04 03:00:00+00:00,5,True,mcx:COPPER30SEP26FUT,1382.3,0.005792,NaN,NaN,2026-09-03 18:00:00+00:00
11543,copper,2026-09-04 03:00:00+00:00,10,True,mcx:COPPER30SEP26FUT,1382.3,0.005336,NaN,NaN,2026-09-03 18:00:00+00:00
12986,copper,2026-09-04 03:00:00+00:00,20,True,mcx:COPPER30OCT26FUT,1389.9,0.005441,NaN,NaN,2026-09-03 18:00:00+00:00
14429,crude,2026-09-04 03:00:00+00:00,5,True,mcx:CRUDEOIL21SEP26FUT,8643.0,0.020563,NaN,NaN,2026-09-03 18:00:00+00:00
15872,crude,2026-09-04 03:00:00+00:00,10,True,mcx:CRUDEOIL21SEP26FUT,8643.0,0.019902,NaN,NaN,2026-09-03 18:00:00+00:00
17315,crude,2026-09-04 03:00:00+00:00,20,True,mcx:CRUDEOIL19OCT26FUT,8355.0,0.017367,NaN,NaN,2026-09-03 18:00:00+00:00
1442,gold,2026-09-04 03:00:00+00:00,5,True,mcx:GOLD05OCT26FUT,155775.0,0.013490,NaN,NaN,2026-09-03 18:00:00+00:00
2885,gold,2026-09-04 03:00:00+00:00,10,True,mcx:GOLD05OCT26FUT,155775.0,0.013490,NaN,NaN,2026-09-03 18:00:00+00:00
4328,gold,2026-09-04 03:00:00+00:00,20,True,mcx:GOLD05OCT26FUT,155775.0,0.013490,19.0,mcx:GOLD04DEC26FUT,2026-09-03 18:00:00+00:00
5771,silver,2026-09-04 03:00:00+00:00,5,True,mcx:SILVER04DEC26FUT,242349.0,0.016195,NaN,NaN,2026-09-03 18:00:00+00:00


## The 2.4 labels ship for research, and are no longer the scored task

`labels/swing_labels` and `labels/return_targets` are the one-sigma barrier labels of rules 2.4. They are still in the pack and still useful to train on, but rules 2.6 grades a submission on the signals it issues against the price path of the designated contract, not on a label. A swing label becomes available at the close of the session that decided it, which is why the latest ones visible at a cutoff belong to cutoffs several sessions earlier.

In [5]:
labels = frames["labels/swing_labels"].collect(columns=["asset", "cutoff_utc", "horizon", "k_multiplier", "label", "hit_session", "available_utc"])
labels = labels[labels["k_multiplier"] == 1.0]
print("scored-multiplier label rows visible:", len(labels))
print("latest label availability:", labels["available_utc"].max().isoformat())
print("latest cutoff with a visible h=5 label:", labels.loc[labels["horizon"] == 5, "cutoff_utc"].max())
print("latest cutoff with a visible h=10 label:", labels.loc[labels["horizon"] == 10, "cutoff_utc"].max())
labels.groupby(["horizon", "label"], observed=True).size().unstack(fill_value=0)

scored-multiplier label rows visible: 17181
latest label availability: 2026-09-03T18:00:00+00:00
latest cutoff with a visible h=5 label: 2026-09-01 03:00:00+00:00
latest cutoff with a visible h=10 label: 2026-09-01 03:00:00+00:00


label,both,down_swing,no_swing,up_swing
horizon,,,,
5,5,1550,2336,1853
10,0,1529,2244,1966
20,0,1513,2080,2105


In [6]:
targets = frames["labels/return_targets"].collect(columns=["asset", "cutoff_utc", "horizon", "y_ret", "y_mfe", "y_mae", "available_utc"])
print("return target rows visible:", len(targets), "latest availability:", targets["available_utc"].max().isoformat())
targets.sort_values("available_utc").tail(6)

return target rows visible: 17131 latest availability: 2026-09-03T18:00:00+00:00


,asset,cutoff_utc,horizon,y_ret,y_mfe,y_mae,available_utc
10000,copper,2026-08-28 03:00:00+00:00,5,-0.008680,0.003901,-0.024135,2026-09-03 18:00:00+00:00
11429,copper,2026-08-21 03:00:00+00:00,10,0.002825,0.021176,-0.012630,2026-09-03 18:00:00+00:00
12848,copper,2026-08-07 03:00:00+00:00,20,-0.002421,0.015930,-0.017876,2026-09-03 18:00:00+00:00
14282,crude,2026-08-28 03:00:00+00:00,5,0.081818,0.098797,-0.012890,2026-09-03 18:00:00+00:00
5716,silver,2026-08-28 03:00:00+00:00,5,-0.018313,0.021545,-0.062769,2026-09-03 18:00:00+00:00
17130,crude,2026-08-07 03:00:00+00:00,20,0.169561,0.186539,-0.012275,2026-09-03 18:00:00+00:00


## Two minutes around a data release

US nonfarm payrolls for August 2026 were released on 4 September 2026 at 12:30 UTC. Compare `calendar/release_surprises` at 12:29 UTC and 12:31 UTC: the release row exists only in the second view.

In [7]:
before = datetime(2026, 9, 4, 12, 29, tzinfo=UTC)
after = datetime(2026, 9, 4, 12, 31, tzinfo=UTC)
columns = ["event_type", "measure", "period_ref", "release_utc", "actual", "consensus", "consensus_quality", "surprise", "surprise_z", "available_utc"]
surprises_before = pit.frame("calendar/release_surprises", before).collect(columns=columns)
surprises_after = pit.frame("calendar/release_surprises", after).collect(columns=columns)
print("release_surprises rows at 12:29Z:", len(surprises_before), " at 12:31Z:", len(surprises_after))
new_rows = surprises_after.merge(surprises_before, how="left", indicator=True).query("_merge == 'left_only'").drop(columns="_merge")
print("rows that appear between the two instants:", len(new_rows))
new_rows

release_surprises rows at 12:29Z: 1834  at 12:31Z: 1837


rows that appear between the two instants: 3


,event_type,measure,period_ref,release_utc,actual,consensus,consensus_quality,surprise,surprise_z,available_utc
1410,us_nfp,ahe_mom,2026-08,2026-09-04 12:30:00+00:00,0.3,0.3,unvintaged,0.0,0.000000,2026-09-04 12:30:00+00:00
1478,us_nfp,nfp_change,2026-08,2026-09-04 12:30:00+00:00,162.0,55.0,unvintaged,107.0,1.212721,2026-09-04 12:30:00+00:00
1546,us_nfp,unemployment_rate,2026-08,2026-09-04 12:30:00+00:00,4.1,4.1,unvintaged,0.0,0.000000,2026-09-04 12:30:00+00:00


In [8]:
vintage_columns = ["event_type", "measure", "period_ref", "kind", "value", "unit", "source", "clock_quality", "captured_utc", "available_utc"]
vint_before = pit.frame("calendar/release_vintages", before).collect(columns=vintage_columns)
vint_after = pit.frame("calendar/release_vintages", after).collect(columns=vintage_columns)
print("release_vintages rows at 12:29Z:", len(vint_before), " at 12:31Z:", len(vint_after))
vint_after.merge(vint_before, how="left", indicator=True).query("_merge == 'left_only'").drop(columns="_merge")

release_vintages rows at 12:29Z: 9406  at 12:31Z: 9423


,event_type,measure,period_ref,kind,value,unit,source,clock_quality,captured_utc,available_utc
6559,us_nfp,ahe_mom,2026-07,revision,0.2,%,investing_history,published,2026-09-15 12:02:01.481954+00:00,2026-09-04 12:30:00+00:00
6562,us_nfp,ahe_mom,2026-07,revision,0.2,%,alfred,published,2026-09-25 21:50:56.147667+00:00,2026-09-04 12:30:00+00:00
6563,us_nfp,ahe_mom,2026-08,actual,0.3,%,investing_history,published,2026-09-15 12:02:01.481954+00:00,2026-09-04 12:30:00+00:00
6564,us_nfp,ahe_mom,2026-08,previous,0.2,%,investing_history,published,2026-09-15 12:02:01.481954+00:00,2026-09-04 12:30:00+00:00
6565,us_nfp,ahe_mom,2026-08,actual,0.3,%,alfred,published,2026-09-25 21:50:56.147667+00:00,2026-09-04 12:30:00+00:00
6566,us_nfp,ahe_mom,2026-08,previous,0.2,%,alfred,published,2026-09-25 21:50:56.147667+00:00,2026-09-04 12:30:00+00:00
7201,us_nfp,nfp_change,2026-06,revision,31.0,K,alfred,published,2026-09-25 21:51:02.254482+00:00,2026-09-04 12:30:00+00:00
7204,us_nfp,nfp_change,2026-07,revision,21.0,K,investing_history,published,2026-09-15 12:02:01.481954+00:00,2026-09-04 12:30:00+00:00
7207,us_nfp,nfp_change,2026-07,revision,21.0,K,alfred,published,2026-09-25 21:51:02.254482+00:00,2026-09-04 12:30:00+00:00
7208,us_nfp,nfp_change,2026-08,actual,162.0,K,investing_history,published,2026-09-15 12:02:01.481954+00:00,2026-09-04 12:30:00+00:00


The same two instants show nothing new for the MCX session of 4 September: its Bhav Copy rows carry the session close as their availability instant, which is later in the day.

In [9]:
def rows_for_session(asof: datetime, session_date) -> int:
    frame = pit.frame("mcx/mcx_daily", asof).collect(columns=["session_date", "asset"])
    return int(((frame["session_date"] == session_date) & frame["asset"].notna()).sum())

session = pd.Timestamp("2026-09-04").date()
next_morning = datetime(2026, 9, 5, 0, 0, tzinfo=UTC)
print("mcx_daily rows for 2026-09-04 at 12:29Z:", rows_for_session(before, session))
print("mcx_daily rows for 2026-09-04 at 12:31Z:", rows_for_session(after, session))
print("mcx_daily rows for 2026-09-04 at 2026-09-05 00:00Z:", rows_for_session(next_morning, session))

mcx_daily rows for 2026-09-04 at 12:29Z: 0
mcx_daily rows for 2026-09-04 at 12:31Z: 0
mcx_daily rows for 2026-09-04 at 2026-09-05 00:00Z: 23


## Unvintaged rows need an explicit opt-in

Rows without a credible availability clock are excluded by default. `include_unvintaged=True` admits them, bounded by the UTC day the feed wrote them where that is known. A submission that uses them must say so in its memo, and the read log records every read that opted in.

In [10]:
name = "pack/quotes/reference_quotes"
strict = pit.frame(name, after).table(columns=["clock_quality"]).num_rows
loose = pit.frame(name, after, include_unvintaged=True).table(columns=["clock_quality"]).num_rows
print(f"{name}: {strict} rows strictly available at 12:31Z, {loose} with unvintaged rows admitted")
schedule_strict = pit.frame("calendar/release_schedule", after).table().num_rows
schedule_loose = pit.frame("calendar/release_schedule", after, include_unvintaged=True).table().num_rows
print(f"calendar/release_schedule: {schedule_strict} strictly available, {schedule_loose} with unvintaged rows admitted")

pack/quotes/reference_quotes: 4137305 rows strictly available at 12:31Z, 4137359 with unvintaged rows admitted
calendar/release_schedule: 2734 strictly available, 3288 with unvintaged rows admitted


## Where a signal may fire

Rules 2.6 section 3 lets a signal fire at any instant inside a published MCX session or at the 08:30 IST pre-open, and section 5 counts a decision record against the sessions of the asset it names. Both questions are answered by the same object the scorer uses: `scoring2.prices.PriceSeries`, which `kit.formats.load_sessions` builds from the pack.

In [11]:
from swing_dataset.kit import THRESHOLD_PCT, formats
from swing_dataset.scoring2 import score as S

prices = formats.load_sessions(PACK_ROOT)
window = S.resolve_window("holdout", prices)
print("holdout window:", window.start.isoformat(), "to", window.end.isoformat())
rows = []
for asset in prices.assets():
    sessions = prices.sessions_between(asset, window.start, window.end)
    pos = prices.session_pos(asset, pd.Timestamp(sessions[-1].isoformat() + "T12:00:00", tz=IST))
    preopen, close = prices.session_bounds(asset, pos)
    rows.append(
        {
            "asset": asset,
            "threshold %": THRESHOLD_PCT[asset],
            "sessions in the holdout": len(sessions),
            "last session": sessions[-1],
            "pre-open (IST)": preopen.tz_convert(IST).strftime("%H:%M"),
            "close (IST)": close.tz_convert(IST).strftime("%H:%M"),
            "designated contract": prices.designated_contract(asset, close),
        }
    )
pd.DataFrame(rows).set_index("asset")

holdout window: 2026-06-01T00:00:00+00:00 to 2026-10-23T18:00:00+00:00


,threshold %,sessions in the holdout,last session,pre-open (IST),close (IST),designated contract
asset,,,,,,
copper,1.5,104,2026-10-23,08:30,23:30,mcx:COPPER30OCT26FUT
crude,3.0,104,2026-10-23,08:30,23:30,mcx:CRUDEOIL19NOV26FUT
gold,1.5,104,2026-10-23,08:30,23:30,mcx:GOLD04DEC26FUT
silver,2.0,104,2026-10-23,08:30,23:30,mcx:SILVER04DEC26FUT


In [12]:
# an instant inside a session, and one that is in none
inside = datetime(2026, 9, 4, 14, 0, tzinfo=IST)
sunday = datetime(2026, 9, 6, 14, 0, tzinfo=IST)
for when in (inside, sunday):
    pos = prices.session_pos("gold", when)
    where = "no session" if pos is None else str(prices.session_dates("gold")[pos])
    print(f"{when.isoformat()}: {where}")
print()
print("a signal at an instant in no session is still counted as issued and still scored;")
print("it simply covers no session, which the coverage floor of section 5 notices.")

2026-09-04T14:00:00+05:30: 2026-09-04
2026-09-06T14:00:00+05:30: no session

a signal at an instant in no session is still counted as issued and still scored;
it simply covers no session, which the coverage floor of section 5 notices.


## Next

`02_baseline_submission.ipynb` runs the five shipped baselines over the holdout, writes their signal files in the submission format, validates them and scores them with the shipped scorer. `template/run.py` does the same as a submission, with the read log the organizers check.